# M.T.S. — Marginal Three-Point Shot

Τελικό notebook για το project. Περιλαμβάνει το fixed-smoothing spline model, το league-wide benchmark και το quadratic regression robustness check.

Αρχεία παράδοσης:

- `group_98_final_report.pdf`
- `group_98_final_notebook.ipynb`


In [7]:
# Αν λείπουν packages, τρέξε μία φορά:
!pip install nba_api scipy pandas numpy matplotlib

import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder, leaguedashplayerstats

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)


## 1. Ρυθμίσεις

Το τελικό report βασίστηκε σε fixed smoothing: `s_mean=0.01`, `s_upper=0.02`.


In [23]:
GROUP_NUMBER = '98'
TARGET_SEASON = '2024-25'
FIT_SEASONS = ['2020-21','2021-22','2022-23','2023-24','2024-25']
SEASON_TYPE = 'Regular Season'

OUTPUT_DIR = Path.home() / 'Documents' / 'nba_team_ppso_outputs_2024_25_FIXED_SMOOTHING_FINAL'
TEAM_PLOTS_DIR = OUTPUT_DIR / 'team_plots'
TEAM_CSV_DIR = OUTPUT_DIR / 'team_csv'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEAM_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TEAM_CSV_DIR.mkdir(parents=True, exist_ok=True)

# Free throw rates used in PPSO
R3 = 0.0331
R2 = 0.3353

ANCHOR_CUTOFF = 0.50
ROLL_WINDOW = 5
SPLINE_K = 3
FIXED_S_MEAN = 0.01
FIXED_S_UPPER = 0.02
SENS_S_GRID = [0.0005,0.001,0.002,0.004,0.008,0.016,0.032]

API_SLEEP = 0.75
API_RETRIES = 4

THRESHOLD_VERY_LOW_KISH = 100
THRESHOLD_LOW_KISH = 150
THRESHOLD_SPARSE_NEAR_OPTIMUM = 5
THRESHOLD_XGT50 = 10
THRESHOLD_XGT55 = 5

# None = all teams. Example: ['Memphis Grizzlies','Toronto Raptors']
TEAMS_TO_RUN = None


## 2. Βοηθητικές συναρτήσεις


In [22]:
def safe_api_call(func, *args, **kwargs):
    last_err = None
    for attempt in range(API_RETRIES):
        try:
            ans = func(*args, **kwargs)
            time.sleep(API_SLEEP)
            return ans
        except Exception as e:
            last_err = e
            print(f'[Retry {attempt+1}/{API_RETRIES}] API error: {e}')
            time.sleep(2 + attempt)
    raise last_err


def safe_name(name):
    return name.replace(' ','_').replace('/','_').replace('.','').replace("'",'').replace('-','_')


def weighted_average(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    m = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if m.sum() == 0:
        return np.nan
    return float(np.sum(values[m] * weights[m]) / np.sum(weights[m]))


def weighted_quantile(values, q, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    m = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values, weights = values[m], weights[m]
    if len(values) == 0:
        return np.nan
    order = np.argsort(values)
    values, weights = values[order], weights[order]
    cutoff = q * weights.sum()
    idx = min(np.searchsorted(np.cumsum(weights), cutoff, side='left'), len(values)-1)
    return float(values[idx])


def kish_effective_size(weights):
    w = np.asarray(weights, dtype=float)
    w = w[np.isfinite(w) & (w > 0)]
    if len(w) == 0:
        return 0.0
    s1, s2 = w.sum(), np.sum(w**2)
    return float((s1**2)/s2) if s2 > 0 else 0.0


def classify_boundary(xstar, xmin, xmax, tol=0.75):
    if abs(xstar - xmin) <= tol:
        return 'left'
    if abs(xstar - xmax) <= tol:
        return 'right'
    return 'interior'


def support_kish(df, mask):
    if mask.sum() == 0:
        return 0.0
    return kish_effective_size(df.loc[mask, 'game_weight'])


def determine_reliability(flags):
    danger = {
        'very_low_total_kish_sample',
        'mean_optimum_left_boundary','mean_optimum_right_boundary',
        'upper_optimum_left_boundary','upper_optimum_right_boundary',
        'sparse_near_mean_optimum','sparse_near_upper_optimum',
        'left_edge_sparse_x_lt_30','right_edge_sparse_x_gt_50','right_edge_very_sparse_x_gt_55'
    }
    if any(f in danger for f in flags):
        return 'DANGER'
    if flags:
        return 'WARNING'
    return 'OK'


## 3. NBA data και βάρη συνέχειας ρόστερ


In [21]:
def get_all_teams():
    all_teams = sorted(teams.get_teams(), key=lambda t: t['full_name'])
    if TEAMS_TO_RUN is not None:
        wanted = set(TEAMS_TO_RUN)
        all_teams = [t for t in all_teams if t['full_name'] in wanted]
    return all_teams


def fetch_team_games(team_id, season):
    gf = safe_api_call(
        leaguegamefinder.LeagueGameFinder,
        team_id_nullable=team_id,
        season_nullable=season,
        season_type_nullable=SEASON_TYPE,
        league_id_nullable='00'
    )
    df = gf.get_data_frames()[0].copy()
    if df.empty:
        return df
    if 'TEAM_ID' in df.columns:
        df = df[df['TEAM_ID'] == team_id].copy()
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    return df.sort_values('GAME_DATE').reset_index(drop=True)


def fetch_player_profile(team_id, season):
    try:
        stats = safe_api_call(
            leaguedashplayerstats.LeagueDashPlayerStats,
            team_id_nullable=team_id,
            season=season,
            season_type_all_star=SEASON_TYPE,
            per_mode_detailed='Totals',
            measure_type_detailed_defense='Base'
        )
        df = stats.get_data_frames()[0].copy()
        for col in ['MIN','FG3A','FGA','PTS','AST','REB','TOV','FG3M','FGM','FTA','FTM']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        if 'PLAYER_ID' in df.columns:
            df['PLAYER_ID'] = df['PLAYER_ID'].astype(int)
        return df
    except Exception as e:
        print(f'Player profile fetch failed ({season}): {e}')
        return pd.DataFrame()


def normalize_share(s):
    total = s.sum()
    return s*0 if total <= 0 else s/total


def compute_c_id(past_df, target_df):
    if past_df.empty or target_df.empty:
        return 0.0
    past = past_df[['PLAYER_ID','FG3A']].copy()
    target = target_df[['PLAYER_ID','FG3A']].copy()
    past['share_past'] = normalize_share(past['FG3A'])
    target['share_target'] = normalize_share(target['FG3A'])
    m = pd.merge(past[['PLAYER_ID','share_past']], target[['PLAYER_ID','share_target']], on='PLAYER_ID', how='outer').fillna(0)
    return float(np.clip(1 - 0.5*np.abs(m['share_past']-m['share_target']).sum(), 0, 1))


def compute_c_role(past_df, target_df, n_bins=5):
    if past_df.empty or target_df.empty:
        return 0.0
    def role_vector(df):
        d = df.sort_values('FG3A', ascending=False).reset_index(drop=True)
        if d['FG3A'].sum() <= 0:
            return np.zeros(n_bins)
        d['share'] = d['FG3A']/d['FG3A'].sum()
        vec = [d.loc[i,'share'] if i < len(d) else 0.0 for i in range(n_bins-1)]
        vec.append(d.loc[n_bins-1:, 'share'].sum() if len(d) >= n_bins else 0.0)
        return np.array(vec)
    return float(np.clip(1 - 0.5*np.abs(role_vector(past_df)-role_vector(target_df)).sum(), 0, 1))


def compute_anchor_scores(df):
    d = df.copy()
    comps = []
    for col in ['MIN','FGA','PTS','AST']:
        comps.append(normalize_share(d[col]) if col in d.columns and d[col].sum() > 0 else pd.Series(np.zeros(len(d)), index=d.index))
    d['anchor_score'] = sum(comps)/len(comps)
    return d[['PLAYER_ID','anchor_score']]


def compute_c_anchor(past_df, target_df):
    if past_df.empty or target_df.empty:
        return 0.0
    past_ids = set(past_df['PLAYER_ID'].astype(int).tolist())
    a = compute_anchor_scores(target_df)
    if a.empty or a['anchor_score'].sum() <= 0:
        return 0.0
    a['anchor_share'] = normalize_share(a['anchor_score'])
    return float(np.clip(a[a['PLAYER_ID'].isin(past_ids)]['anchor_share'].sum(), 0, 1))


def compute_season_weights(team_id, team_name):
    profiles = {}
    for season in FIT_SEASONS:
        print(f'Fetching player profile: {team_name}, {season}')
        profiles[season] = fetch_player_profile(team_id, season)
    target_profile = profiles[TARGET_SEASON]
    rows, weights = [], {}
    for season in FIT_SEASONS:
        if season == TARGET_SEASON:
            c_id = c_role = c_anchor = final_weight = 1.0
        else:
            c_id = compute_c_id(profiles[season], target_profile)
            c_role = compute_c_role(profiles[season], target_profile)
            c_anchor = compute_c_anchor(profiles[season], target_profile)
            raw = min(c_anchor, 0.3*c_id + 0.7*c_role)
            final_weight = 0.0 if c_anchor < ANCHOR_CUTOFF else raw
        weights[season] = float(final_weight)
        rows.append({'TEAM_NAME':team_name,'SEASON':season,'C_ID_shooters':c_id,'C_ROLE_shooting_structure':c_role,'C_ANCHOR_core':c_anchor,'final_weight':final_weight})
    return weights, pd.DataFrame(rows)


## 4. Game-level μεταβλητές


In [20]:
def prepare_game_data(raw_df, season, season_weight):
    df = raw_df.copy()
    for col in ['FGA','FGM','FG3A','FG3M','FTA','FTM','TOV']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['FG2A'] = df['FGA'] - df['FG3A']
    df['FG2M'] = df['FGM'] - df['FG3M']
    df['x'] = np.where(df['FGA'] > 0, 100*df['FG3A']/df['FGA'], np.nan)
    df['q3'] = np.where(df['FG3A'] > 0, df['FG3M']/df['FG3A'], np.nan)
    df['q2'] = np.where(df['FG2A'] > 0, df['FG2M']/df['FG2A'], np.nan)
    df['qf'] = np.where(df['FTA'] > 0, df['FTM']/df['FTA'], np.nan)
    df['TOV_per100FGA'] = np.where(df['FGA'] > 0, 100*df['TOV']/df['FGA'], np.nan)
    df['FTA_per100FGA'] = np.where(df['FGA'] > 0, 100*df['FTA']/df['FGA'], np.nan)
    df['FTA_other_per100FGA'] = (df['FTA_per100FGA'] - R3*df['x'] - R2*(100-df['x'])).clip(lower=0)
    df['SEASON'] = season
    df['season_weight'] = float(season_weight)
    df['game_weight'] = float(season_weight)
    return df


def build_team_dataset(team_id, team_name, season_weights):
    parts, target_df = [], None
    for season in FIT_SEASONS:
        print(f'Fetching game logs: {team_name}, {season}')
        raw = fetch_team_games(team_id, season)
        if raw.empty:
            continue
        p = prepare_game_data(raw, season, season_weights.get(season, 0.0))
        if season == TARGET_SEASON:
            target_df = p.copy()
        parts.append(p)
    if not parts:
        return pd.DataFrame(), pd.DataFrame()
    full = pd.concat(parts, ignore_index=True)
    full = full[np.isfinite(full['x']) & (full['game_weight'] > 0)].copy()
    return full, target_df


def compute_target_constants(target_df):
    current_x = float(target_df['x'].mean())
    qf = float(target_df['qf'].dropna().mean()) if target_df['qf'].notna().any() else 0.78
    TOV = float(target_df['TOV_per100FGA'].dropna().mean())
    FTA_other = float(target_df['FTA_other_per100FGA'].dropna().mean())
    return current_x, qf, TOV, FTA_other


## 5. Smoothing splines και PPSO


In [19]:
def rolling_points(df, y_col, window=ROLL_WINDOW):
    d = df[np.isfinite(df[y_col]) & np.isfinite(df['x']) & (df['game_weight'] > 0)].sort_values('x').reset_index(drop=True)
    if len(d) < window:
        return pd.DataFrame()
    xs, ys, ws = d['x'].to_numpy(), d[y_col].to_numpy(), d['game_weight'].to_numpy()
    rows, half = [], window//2
    for i in range(len(d)):
        lo, hi = max(0, i-half), min(len(d), i+half+1)
        xw, yw, ww = xs[lo:hi], ys[lo:hi], ws[lo:hi]
        m = np.isfinite(xw) & np.isfinite(yw) & np.isfinite(ww) & (ww > 0)
        if m.sum() < 2:
            continue
        rows.append({'avg_x': weighted_average(xw[m], ww[m]), 'weighted_mean': weighted_average(yw[m], ww[m]), 'weighted_q90': weighted_quantile(yw[m], 0.90, ww[m]), 'window_weight': ww[m].sum(), 'window_kish': kish_effective_size(ww[m])})
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out['x_round'] = out['avg_x'].round(4)
    grouped = []
    for _, g in out.groupby('x_round'):
        grouped.append({'avg_x': weighted_average(g['avg_x'], g['window_weight']), 'weighted_mean': weighted_average(g['weighted_mean'], g['window_weight']), 'weighted_q90': weighted_average(g['weighted_q90'], g['window_weight']), 'window_weight': g['window_weight'].sum(), 'window_kish': g['window_kish'].sum()})
    return pd.DataFrame(grouped).sort_values('avg_x').reset_index(drop=True)


def build_bin_metrics(full_df):
    q3 = rolling_points(full_df, 'q3').rename(columns={'weighted_mean':'roll5_weighted_q3','weighted_q90':'roll5_q90_q3','window_weight':'q3_window_weight','window_kish':'q3_window_kish'})
    q2 = rolling_points(full_df, 'q2').rename(columns={'weighted_mean':'roll5_weighted_q2','weighted_q90':'roll5_q90_q2','window_weight':'q2_window_weight','window_kish':'q2_window_kish'})
    if q3.empty or q2.empty:
        return pd.DataFrame()
    n = min(len(q3), len(q2))
    b = pd.DataFrame({'avg_x':(q3['avg_x'].iloc[:n].to_numpy()+q2['avg_x'].iloc[:n].to_numpy())/2, 'roll5_weighted_q3':q3['roll5_weighted_q3'].iloc[:n].to_numpy(), 'roll5_q90_q3':q3['roll5_q90_q3'].iloc[:n].to_numpy(), 'roll5_weighted_q2':q2['roll5_weighted_q2'].iloc[:n].to_numpy(), 'roll5_q90_q2':q2['roll5_q90_q2'].iloc[:n].to_numpy(), 'q3_window_weight':q3['q3_window_weight'].iloc[:n].to_numpy(), 'q2_window_weight':q2['q2_window_weight'].iloc[:n].to_numpy()})
    b['fit_weight'] = (b['q3_window_weight'] + b['q2_window_weight'])/2
    return b.dropna().sort_values('avg_x').reset_index(drop=True)


def fit_spline(x, y, w, s_multiplier):
    tmp = pd.DataFrame({'x':np.asarray(x,float), 'y':np.asarray(y,float), 'w':np.asarray(w,float)}).dropna()
    tmp = tmp[tmp['w'] > 0].sort_values('x')
    tmp = tmp.groupby('x', as_index=False).agg({'y':'mean','w':'sum'})
    if len(tmp) < 5:
        return None
    k_use = min(SPLINE_K, len(tmp)-1)
    return UnivariateSpline(tmp['x'], tmp['y'], w=np.sqrt(tmp['w']), s=s_multiplier*len(tmp), k=k_use)


def make_ppso_function(S3, S2, qf, TOV, FTA_other, xmin, xmax):
    def ppso(x):
        x = np.asarray(x, dtype=float)
        xc = np.clip(x, xmin, xmax)
        num = xc*(3*S3(xc) + qf*R3) + (100-xc)*(2*S2(xc) + qf*R2) + qf*FTA_other
        den = (1+R3)*xc + (1+R2)*(100-xc) + FTA_other + TOV
        return num/den
    return ppso


def optimize_ppso(ppso, xmin, xmax):
    xg = np.linspace(xmin, xmax, 1000)
    yg = ppso(xg)
    i = int(np.nanargmax(yg))
    xs = float(xg[i])
    return xs, float(yg[i]), classify_boundary(xs, xmin, xmax)


def numeric_derivative(f, x, xmin, xmax, h=1e-3):
    x1, x2 = max(xmin, x-h), min(xmax, x+h)
    return float((f(x2)-f(x1))/(x2-x1)) if x2 > x1 else np.nan


## 6. Sensitivity και διαγράμματα


In [18]:
def sensitivity_analysis(bin_df, qf, TOV, FTA_other, xmin, xmax):
    rows = []
    x = bin_df['avg_x'].to_numpy()
    w = bin_df['fit_weight'].to_numpy()
    xg = np.linspace(xmin, xmax, 800)
    for s in SENS_S_GRID:
        S3m = fit_spline(x, bin_df['roll5_weighted_q3'], w, s)
        S3u = fit_spline(x, bin_df['roll5_q90_q3'], w, s)
        S2m = fit_spline(x, bin_df['roll5_weighted_q2'], w, s)
        S2u = fit_spline(x, bin_df['roll5_q90_q2'], w, s)
        if any(sp is None for sp in [S3m,S3u,S2m,S2u]):
            rows.append({'s':s, 'mean_xstar':np.nan, 'upper_xstar':np.nan})
            continue
        pm = make_ppso_function(S3m, S2m, qf, TOV, FTA_other, xmin, xmax)
        pu = make_ppso_function(S3u, S2u, qf, TOV, FTA_other, xmin, xmax)
        rows.append({'s':s, 'mean_xstar':float(xg[np.nanargmax(pm(xg))]), 'upper_xstar':float(xg[np.nanargmax(pu(xg))])})
    return pd.DataFrame(rows)


def make_basic_plots(team_name, plot_dir, bin_df, current_x, S3m, S3u, S2m, S2u, pm, pu, xmin, xmax, mx, ux, mmax, umax, sens_df):
    plot_dir.mkdir(parents=True, exist_ok=True)
    team_safe = safe_name(team_name)
    xg = np.linspace(xmin, xmax, 700)
    # PPSO plot
    fig, ax = plt.subplots(figsize=(10,6))
    ax.plot(xg, pm(xg), label='Mean PPSO(x)', linewidth=2.5)
    ax.plot(xg, pu(xg), label='Upper PPSO(x)', linewidth=2.5)
    ax.axvline(current_x, linestyle=':', linewidth=2, label=f'Current x = {current_x:.2f}')
    ax.axvline(mx, linestyle='--', linewidth=2, label=f'Mean x* = {mx:.2f}')
    ax.axvline(ux, linestyle='-.', linewidth=2, label=f'Upper x* = {ux:.2f}')
    ax.scatter([current_x], [float(pm(current_x))], s=90, label=f'Current mean PPSO = {float(pm(current_x)):.4f}')
    ax.scatter([mx], [mmax], s=90, label=f'Mean max PPSO = {mmax:.4f}')
    ax.scatter([ux], [umax], s=90, label=f'Upper max PPSO = {umax:.4f}')
    ax.set_title(f'{team_name}: PPSO(x) [fixed smoothing]')
    ax.set_xlabel('x = 3PA per 100 FGA')
    ax.set_ylabel('PPSO')
    ax.grid(True, alpha=.35)
    ax.legend()
    fig.tight_layout()
    ppso_path = plot_dir / f'{team_safe}_PPSO_spline.png'
    fig.savefig(ppso_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    # sensitivity
    fig, ax = plt.subplots(figsize=(10,5))
    v = sens_df.dropna()
    if not v.empty:
        ax.plot(v['s'], v['mean_xstar'], 'o-', label='Mean x*(s)')
        ax.plot(v['s'], v['upper_xstar'], 's-', label='Upper x*(s)')
    ax.axvline(FIXED_S_MEAN, linestyle='--', label=f'Fixed mean s={FIXED_S_MEAN}')
    ax.axvline(FIXED_S_UPPER, linestyle='-.', label=f'Fixed upper s={FIXED_S_UPPER}')
    ax.set_xscale('log')
    ax.set_title(f'{team_name}: Sensitivity of x* to smoothing parameter s')
    ax.set_xlabel('s multiplier, log scale')
    ax.set_ylabel('Optimal x*')
    ax.grid(True, alpha=.35)
    ax.legend()
    fig.tight_layout()
    sens_path = plot_dir / f'{team_safe}_sensitivity_s.png'
    fig.savefig(sens_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    return ppso_path, sens_path


## 7. Team-specific fixed-smoothing model


In [17]:
def process_team(team):
    team_id, team_name = team['id'], team['full_name']
    team_safe = safe_name(team_name)
    print('' + '='*60)
    print('TEAM:', team_name)
    print('='*60)
    season_weights, continuity_df = compute_season_weights(team_id, team_name)
    full_df, target_df = build_team_dataset(team_id, team_name, season_weights)
    if full_df.empty or target_df.empty:
        return None, continuity_df
    current_x, qf, TOV, FTA_other = compute_target_constants(target_df)
    bin_df = build_bin_metrics(full_df)
    if bin_df.empty or len(bin_df) < 8:
        return None, continuity_df
    xmin, xmax = float(bin_df['avg_x'].min()), float(bin_df['avg_x'].max())
    x, w = bin_df['avg_x'].to_numpy(), bin_df['fit_weight'].to_numpy()
    S3m = fit_spline(x, bin_df['roll5_weighted_q3'], w, FIXED_S_MEAN)
    S3u = fit_spline(x, bin_df['roll5_q90_q3'], w, FIXED_S_UPPER)
    S2m = fit_spline(x, bin_df['roll5_weighted_q2'], w, FIXED_S_MEAN)
    S2u = fit_spline(x, bin_df['roll5_q90_q2'], w, FIXED_S_UPPER)
    if any(sp is None for sp in [S3m,S3u,S2m,S2u]):
        return None, continuity_df
    pm = make_ppso_function(S3m, S2m, qf, TOV, FTA_other, xmin, xmax)
    pu = make_ppso_function(S3u, S2u, qf, TOV, FTA_other, xmin, xmax)
    mx, mmax, mb = optimize_ppso(pm, xmin, xmax)
    ux, umax, ub = optimize_ppso(pu, xmin, xmax)
    sens_df = sensitivity_analysis(bin_df, qf, TOV, FTA_other, xmin, xmax)
    sens_mean_std, sens_upper_std = float(sens_df['mean_xstar'].std()), float(sens_df['upper_xstar'].std())
    kish_total = kish_effective_size(full_df['game_weight'])
    kish_gt_50 = support_kish(full_df, full_df['x'] > 50)
    kish_gt_55 = support_kish(full_df, full_df['x'] > 55)
    kish_lt_30 = support_kish(full_df, full_df['x'] < 30)
    kish_near_mean = support_kish(full_df, np.abs(full_df['x'] - mx) <= 1.0)
    kish_near_upper = support_kish(full_df, np.abs(full_df['x'] - ux) <= 1.0)
    flags = []
    if kish_total < THRESHOLD_VERY_LOW_KISH: flags.append('very_low_total_kish_sample')
    elif kish_total < THRESHOLD_LOW_KISH: flags.append('low_total_kish_sample')
    if mb == 'left': flags.append('mean_optimum_left_boundary')
    elif mb == 'right': flags.append('mean_optimum_right_boundary')
    if ub == 'left': flags.append('upper_optimum_left_boundary')
    elif ub == 'right': flags.append('upper_optimum_right_boundary')
    if kish_near_mean < THRESHOLD_SPARSE_NEAR_OPTIMUM: flags.append('sparse_near_mean_optimum')
    if kish_near_upper < THRESHOLD_SPARSE_NEAR_OPTIMUM: flags.append('sparse_near_upper_optimum')
    if (mb == 'left' or ub == 'left') and kish_lt_30 < THRESHOLD_XGT50: flags.append('left_edge_sparse_x_lt_30')
    if (mb == 'right' or ub == 'right') and kish_gt_50 < THRESHOLD_XGT50: flags.append('right_edge_sparse_x_gt_50')
    if (mb == 'right' or ub == 'right') and kish_gt_55 < THRESHOLD_XGT55: flags.append('right_edge_very_sparse_x_gt_55')
    if abs(mx - ux) > 8: flags.append('mean_upper_scenarios_disagree')
    if sens_mean_std > 5: flags.append('mean_xstar_high_variance_across_s_grid')
    if sens_upper_std > 5: flags.append('upper_xstar_high_variance_across_s_grid')
    reliability = determine_reliability(flags)
    print(f'Current x: {current_x:.2f}')
    print(f'Mean optimal x: {mx:.2f} | boundary: {mb}')
    print(f'Upper optimal x: {ux:.2f} | boundary: {ub}')
    print('Reliability:', reliability)
    print('Flags:', '; '.join(flags) if flags else 'None')
    team_plot_dir, team_csv_dir = TEAM_PLOTS_DIR/team_safe, TEAM_CSV_DIR/team_safe
    team_plot_dir.mkdir(parents=True, exist_ok=True)
    team_csv_dir.mkdir(parents=True, exist_ok=True)
    bin_df.to_csv(team_csv_dir/f'{team_safe}_bin_metrics.csv', index=False)
    full_df.to_csv(team_csv_dir/f'{team_safe}_weighted_games.csv', index=False)
    sens_df.to_csv(team_csv_dir/f'{team_safe}_sensitivity_s.csv', index=False)
    xg = np.linspace(xmin, xmax, 500)
    pd.DataFrame({'x':xg,'S3_mean':S3m(xg),'S3_upper':S3u(xg),'S2_mean':S2m(xg),'S2_upper':S2u(xg),'PPSO_mean':pm(xg),'PPSO_upper':pu(xg)}).to_csv(team_csv_dir/f'{team_safe}_spline_values.csv', index=False)
    ppso_path, sens_path = make_basic_plots(team_name, team_plot_dir, bin_df, current_x, S3m,S3u,S2m,S2u, pm,pu, xmin,xmax, mx,ux,mmax,umax, sens_df)
    row = {'TEAM_NAME':team_name, 'current_x':current_x, 'qf':qf, 'TOV_per100FGA':TOV, 'FTA_other_per100FGA':FTA_other, 'mean_optimal_x':mx, 'upper_optimal_x':ux, 'mean_boundary':mb, 'upper_boundary':ub, 'current_mean_ppso':float(pm(current_x)), 'current_upper_ppso':float(pu(current_x)), 'mean_max_ppso':mmax, 'upper_max_ppso':umax, 'mean_derivative_current':numeric_derivative(pm, current_x, xmin, xmax), 'upper_derivative_current':numeric_derivative(pu, current_x, xmin, xmax), 'mean_xstar_std_across_s_grid':sens_mean_std, 'upper_xstar_std_across_s_grid':sens_upper_std, 'raw_games':len(full_df), 'weighted_games_sum':float(full_df['game_weight'].sum()), 'kish_effective_games':kish_total, 'sample_kish_x_gt_50':kish_gt_50, 'sample_kish_x_gt_55':kish_gt_55, 'sample_kish_x_lt_30':kish_lt_30, 'kish_near_mean_optimum':kish_near_mean, 'kish_near_upper_optimum':kish_near_upper, 'reliability_level':reliability, 'reliability_flags':'; '.join(flags), 'PPSO_plot_path':str(ppso_path), 'sensitivity_plot_path':str(sens_path)}
    for s in FIT_SEASONS:
        row[f'weight_{s}'] = season_weights.get(s, np.nan)
    return row, continuity_df


def run_team_specific_model():
    rows, conts = [], []
    for team in get_all_teams():
        try:
            row, cont = process_team(team)
            if cont is not None and not cont.empty: conts.append(cont)
            if row is not None: rows.append(row)
        except Exception as e:
            print(f'ERROR for team {team["full_name"]}: {e}')
    summary = pd.DataFrame(rows)
    summary.to_csv(OUTPUT_DIR/'nba_team_specific_ppso_optima_summary.csv', index=False)
    if conts:
        pd.concat(conts, ignore_index=True).to_csv(OUTPUT_DIR/'nba_all_team_continuity_weights.csv', index=False)
    for level in ['DANGER','WARNING','OK']:
        summary[summary['reliability_level']==level].to_csv(OUTPUT_DIR/f'reliability_{level.lower()}_teams.csv', index=False)
    print('Reliability counts:')
    print(summary['reliability_level'].value_counts())
    print('Saved under:', OUTPUT_DIR)
    return summary


In [ ]:
# Τρέξε αυτό για το βασικό team-specific μοντέλο.
# Θέλει αρκετή ώρα λόγω NBA API.
team_summary_df = run_team_specific_model()


## 8. League-wide benchmark


In [24]:
LEAGUE_OUTPUT_DIR = Path.home() / 'Documents' / 'nba_league_wide_ppso_recent_weighted'
LEAGUE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LEAGUE_SEASON_WEIGHTS = {'2020-21':0.50, '2021-22':0.65, '2022-23':0.80, '2023-24':0.95, '2024-25':1.00}

def build_league_dataset():
    parts = []
    for season in FIT_SEASONS:
        sw = LEAGUE_SEASON_WEIGHTS.get(season, 1.0)
        for team in get_all_teams():
            try:
                print(f'Fetching league data: {season}, {team["full_name"]}')
                raw = fetch_team_games(team['id'], season)
                if raw.empty: continue
                p = prepare_game_data(raw, season, sw)
                p['TEAM_NAME_STATIC'] = team['full_name']
                parts.append(p)
            except Exception as e:
                print('Error:', season, team['full_name'], e)
    df = pd.concat(parts, ignore_index=True)
    return df[np.isfinite(df['x']) & (df['game_weight'] > 0)].copy()


def run_league_wide_model():
    df = build_league_dataset().dropna(subset=['x','q2','q3']).copy()
    print('Raw/usable league rows:', len(df))
    print('Weighted games sum:', df['game_weight'].sum())
    print('Kish effective games:', kish_effective_size(df['game_weight']))
    current_x = weighted_average(df['x'], df['game_weight'])
    qf = weighted_average(df['qf'], df['game_weight'])
    TOV = weighted_average(df['TOV_per100FGA'], df['game_weight'])
    FTA_other = weighted_average(df['FTA_other_per100FGA'], df['game_weight'])
    b = build_bin_metrics(df)
    xmin, xmax = float(b['avg_x'].min()), float(b['avg_x'].max())
    x, w = b['avg_x'].to_numpy(), b['fit_weight'].to_numpy()
    S3m = fit_spline(x, b['roll5_weighted_q3'], w, FIXED_S_MEAN)
    S3u = fit_spline(x, b['roll5_q90_q3'], w, FIXED_S_UPPER)
    S2m = fit_spline(x, b['roll5_weighted_q2'], w, FIXED_S_MEAN)
    S2u = fit_spline(x, b['roll5_q90_q2'], w, FIXED_S_UPPER)
    pm = make_ppso_function(S3m, S2m, qf, TOV, FTA_other, xmin, xmax)
    pu = make_ppso_function(S3u, S2u, qf, TOV, FTA_other, xmin, xmax)
    mx, mmax, mb = optimize_ppso(pm, xmin, xmax)
    ux, umax, ub = optimize_ppso(pu, xmin, xmax)
    print(f'Current league x: {current_x:.2f}')
    print(f'Mean x*: {mx:.2f} | boundary: {mb}')
    print(f'Upper x*: {ux:.2f} | boundary: {ub}')
    xg = np.linspace(xmin, xmax, 600)
    curve = pd.DataFrame({'x':xg,'S3_mean':S3m(xg),'S3_upper':S3u(xg),'S2_mean':S2m(xg),'S2_upper':S2u(xg),'PPSO_mean':pm(xg),'PPSO_upper':pu(xg)})
    summary = pd.DataFrame([{'current_x':current_x,'qf':qf,'TOV_per100FGA':TOV,'FTA_other_per100FGA':FTA_other,'x_min':xmin,'x_max':xmax,'mean_xstar':mx,'upper_xstar':ux,'mean_boundary':mb,'upper_boundary':ub,'current_mean_ppso':float(pm(current_x)),'mean_max_ppso':mmax,'current_upper_ppso':float(pu(current_x)),'upper_max_ppso':umax,'weighted_games_sum':float(df['game_weight'].sum()),'kish_effective_games':kish_effective_size(df['game_weight'])}])
    curve.to_csv(LEAGUE_OUTPUT_DIR/'league_wide_recent_weighted_curve_values.csv', index=False)
    b.to_csv(LEAGUE_OUTPUT_DIR/'league_wide_recent_weighted_bin_points.csv', index=False)
    summary.to_csv(LEAGUE_OUTPUT_DIR/'league_wide_recent_weighted_summary.csv', index=False)
    fig, ax = plt.subplots(figsize=(10,6))
    ax.plot(xg, pm(xg), label='Mean PPSO(x)', linewidth=2.5)
    ax.plot(xg, pu(xg), label='Upper PPSO(x)', linewidth=2.5)
    ax.axvline(current_x, linestyle=':', label=f'Current x={current_x:.2f}')
    ax.axvline(mx, linestyle='--', label=f'Mean x*={mx:.2f}')
    ax.axvline(ux, linestyle='-.', label=f'Upper x*={ux:.2f}')
    ax.set_title('League-wide: PPSO(x) [recent weighted]')
    ax.set_xlabel('x = 3PA per 100 FGA')
    ax.set_ylabel('PPSO')
    ax.grid(True, alpha=.35)
    ax.legend()
    fig.tight_layout()
    fig.savefig(LEAGUE_OUTPUT_DIR/'league_wide_PPSO_recent_weighted.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    return summary, b, curve


In [ ]:
# Τρέξε αυτό για το league-wide benchmark.
league_summary_df, league_bin_df, league_curve_df = run_league_wide_model()


## 9. Quadratic regression robustness check


In [25]:
QUAD_OUTPUT_DIR = Path.home() / 'Documents' / 'nba_team_ppso_outputs_2024_25_QUADRATIC_COMPARISON'
QUAD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def fit_weighted_quadratic(x, y, w):
    tmp = pd.DataFrame({'x':x, 'y':y, 'w':w}).dropna()
    tmp = tmp[tmp['w'] > 0]
    if len(tmp) < 5:
        return None
    return np.poly1d(np.polyfit(tmp['x'], tmp['y'], deg=2, w=np.sqrt(tmp['w'])))


def run_quadratic_comparison():
    rows = []
    for team in get_all_teams():
        try:
            team_name = team['full_name']
            sw, _ = compute_season_weights(team['id'], team_name)
            full_df, target_df = build_team_dataset(team['id'], team_name, sw)
            if full_df.empty or target_df.empty: continue
            current_x, qf, TOV, FTA_other = compute_target_constants(target_df)
            q3p = fit_weighted_quadratic(full_df['x'], full_df['q3'], full_df['game_weight'])
            q2p = fit_weighted_quadratic(full_df['x'], full_df['q2'], full_df['game_weight'])
            if q3p is None or q2p is None: continue
            xmin, xmax = float(full_df['x'].min()), float(full_df['x'].max())
            q3f = lambda z: np.clip(q3p(z), 0, 1)
            q2f = lambda z: np.clip(q2p(z), 0, 1)
            ppso = make_ppso_function(q3f, q2f, qf, TOV, FTA_other, xmin, xmax)
            xs, mxv, boundary = optimize_ppso(ppso, xmin, xmax)
            kish_total = kish_effective_size(full_df['game_weight'])
            kish_near = support_kish(full_df, np.abs(full_df['x']-xs) <= 1.0)
            flags = []
            if kish_total < THRESHOLD_VERY_LOW_KISH: flags.append('very_low_total_kish_sample')
            elif kish_total < THRESHOLD_LOW_KISH: flags.append('low_total_kish_sample')
            if boundary == 'left': flags.append('quad_optimum_left_boundary')
            elif boundary == 'right': flags.append('quad_optimum_right_boundary')
            if kish_near < THRESHOLD_SPARSE_NEAR_OPTIMUM: flags.append('sparse_near_quad_optimum')
            reliability = 'DANGER' if any(f in {'very_low_total_kish_sample','quad_optimum_left_boundary','quad_optimum_right_boundary','sparse_near_quad_optimum'} for f in flags) else ('WARNING' if flags else 'OK')
            print(f'{team_name}: current={current_x:.2f}, quad x*={xs:.2f}, boundary={boundary}, quad reliability={reliability}')
            rows.append({'TEAM_NAME':team_name,'current_x':current_x,'quad_xstar':xs,'quad_max_ppso':mxv,'quad_boundary':boundary,'quad_reliability':reliability,'quad_flags':'; '.join(flags),'kish_effective_games':kish_total,'kish_near_quad_optimum':kish_near})
        except Exception as e:
            print('ERROR:', team['full_name'], e)
    summary = pd.DataFrame(rows)
    summary.to_csv(QUAD_OUTPUT_DIR/'quadratic_ppso_summary.csv', index=False)
    print(summary['quad_reliability'].value_counts())
    return summary


In [ ]:
# Τρέξε αυτό για το quadratic regression robustness check.
quadratic_summary_df = run_quadratic_comparison()


## 10. Τελικοί αριθμητικοί έλεγχοι


In [26]:
summary_df = pd.read_csv(OUTPUT_DIR / 'nba_team_specific_ppso_optima_summary.csv')
print('Reliability counts:')
print(summary_df['reliability_level'].value_counts())
print('Mean scenario increase:', (summary_df['mean_optimal_x'] > summary_df['current_x']).sum(), '/30')
print('Upper scenario increase:', (summary_df['upper_optimal_x'] > summary_df['current_x']).sum(), '/30')
print('Mean interior optima:', (summary_df['mean_boundary'] == 'interior').sum(), '/30')
print('Upper interior optima:', (summary_df['upper_boundary'] == 'interior').sum(), '/30')
print('Both scenarios interior:', ((summary_df['mean_boundary'] == 'interior') & (summary_df['upper_boundary'] == 'interior')).sum(), '/30')
sample_problem = summary_df[summary_df['reliability_flags'].fillna('').str.contains('low_total_kish_sample|very_low_total_kish_sample', regex=True)]
print('Teams with total sample size issue:', len(sample_problem))
display(sample_problem[['TEAM_NAME','kish_effective_games','reliability_flags']])


Reliability counts:
reliability_level
DANGER     25
WARNING     3
OK          2
Name: count, dtype: int64
Mean scenario increase: 24 /30
Upper scenario increase: 29 /30
Mean interior optima: 14 /30
Upper interior optima: 13 /30
Both scenarios interior: 9 /30
Teams with total sample size issue: 3


,TEAM_NAME,kish_effective_games,reliability_flags
6,Dallas Mavericks,149.203335,low_total_kish_sample; sparse_near_mean_optimu...
22,Philadelphia 76ers,82.000000,very_low_total_kish_sample; mean_optimum_left_...
29,Washington Wizards,148.880400,low_total_kish_sample; upper_optimum_right_bou...
